# 恢復本經文 → 私人 R2（M2 經文對齊用）

把雲端硬碟 `twgbr-archive/recovery-version/text` 的恢復本經文，整理成每章一個檔（共 1,189 章），上傳到 Cloudflare R2 的私人 bucket
`meeting-captions-scripture`。網站只讓**房間房主**一次讀取綱要用到的幾節，不會公開。CPU 即可，約 2–5 分鐘。

**先在 Colab 左側「🔑 密鑰」加入三個值**（不要貼在對話或筆記本內容裏），並打開「筆記本存取權」：
- `R2_ACCOUNT_ID`：Cloudflare 帳戶 ID
- `R2_ACCESS_KEY_ID`、`R2_SECRET_ACCESS_KEY`：R2 API 權杖（只開這個 bucket 的「物件讀取和寫入」）

跑完把最後幾行（章數、節數、讀回檢查）貼回對話，或回覆「跑完了」。

In [ ]:
# 第 1 格：連接雲端硬碟、找到恢復本文字
import os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
found = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'recovery-version' in dirs]
if not found or not (found[0]/'recovery-version'/'text').is_dir():
    raise RuntimeError('找不到 twgbr-archive/recovery-version/text')
ARCHIVE = found[0]
print('資料：', ARCHIVE)

In [ ]:
# 第 2 格：取得固定版本的程式與套件
COMMIT = 'c94f92efdd470fbf4b58da5aa423f2d644ba83e8'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boto3==1.35.99'], check=True)
print('程式版本', COMMIT[:12])

In [ ]:
# 第 3 格：整理並上傳（先檢查 1,189 章都在才上傳）
env = dict(os.environ)
for name in ('R2_ACCOUNT_ID', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY'):
    try:
        env[name] = userdata.get(name)
    except Exception:
        raise RuntimeError(f'Colab 密鑰缺少 {name}（或沒有打開「筆記本存取權」）')
result = subprocess.run([sys.executable, str(REPO/'scripts/upload_scripture.py'), '--archive', str(ARCHIVE), '--upload'],
                        cwd=REPO, env=env, capture_output=True, text=True)
print(result.stdout[-5000:])
if result.returncode:
    print(result.stderr[-3000:].replace(env['R2_SECRET_ACCESS_KEY'], '***'))
    raise RuntimeError('上傳沒有完成，請看上面的訊息')